In [1]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("../data/cleaned/naukri_data_science_jobs_india_cleaned.csv")

df.head()

,company,location,education,skills_description,job_type,salary,job_experience,job_role
0,UPL,"Bengaluru, Mumbai",Postgraduate,"python, MLT, statistical modeling, machine lea...",contract,939957.0,5-6,Senior Data Scientist
1,Walmart,Bengaluru,Postgraduate,"Data Science, Machine learning, Python, Azure,...",contract,2060767.0,1-7,Senior Data Scientist
2,SAP India Pvt.Ltd,Bengaluru,Undergraduate,"Python, IT Skills, Testing, Cloud, Product Man...",full-time,698970.0,7-8,Applied Data Scientist / ML Senior Engineer (P...
3,UPL,"Bengaluru, Mumbai",Postgraduate,"python, machine learning, Data Science, data a...",full-time,529674.0,6-7,Data Scientist
4,Walmart,Bengaluru,Undergraduate,"IT Skills, Python, Data Science, Machine Learn...",contract,2163829.0,3-7,Data Scientist


In [2]:
# Split experience into minimum and maximum years
df[["exp_min", "exp_max"]] = df["job_experience"].str.split("-", expand=True)

# Convert to integer
df["exp_min"] = df["exp_min"].astype(int)
df["exp_max"] = df["exp_max"].astype(int)

# Average experience
df["exp_avg"] = (df["exp_min"] + df["exp_max"]) / 2

In [3]:
df[["job_experience", "exp_min", "exp_max", "exp_avg"]].head(10)

,job_experience,exp_min,exp_max,exp_avg
0,5-6,5,6,5.5
1,1-7,1,7,4.0
2,7-8,7,8,7.5
3,6-7,6,7,6.5
4,3-7,3,7,5.0
5,6-7,6,7,6.5
6,2-4,2,4,3.0
7,1-4,1,4,2.5
8,3-4,3,4,3.5
9,6-8,6,8,7.0


In [4]:
# Create primary location
df["primary_location"] = (
    df["location"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [5]:
df[["location", "primary_location"]].sample(10, random_state=42)

,location,primary_location
1935,"Hyderabad, Chennai, Bengaluru",Hyderabad
6494,Gurugram,Gurugram
1720,Bengaluru,Bengaluru
9120,Bengaluru,Bengaluru
360,Bengaluru,Bengaluru
9663,Bengaluru,Bengaluru
5277,Bengaluru,Bengaluru
8546,Pune,Pune
2221,Pune,Pune
4617,Nagpur,Nagpur


In [6]:
# Initially copy the original job role
df["role_category"] = df["job_role"]

In [7]:
# Standardize common roles
df.loc[
    df["job_role"].str.contains("data engineer", case=False, na=False),
    "role_category"
] = "Data Engineer"

df.loc[
    df["job_role"].str.contains("data scientist|data science", case=False, na=False),
    "role_category"
] = "Data Scientist"

df.loc[
    df["job_role"].str.contains("data analyst", case=False, na=False),
    "role_category"
] = "Data Analyst"

df.loc[
    df["job_role"].str.contains("business intelligence", case=False, na=False),
    "role_category"
] = "BI Engineer"

df.loc[
    df["job_role"].str.contains("business analyst", case=False, na=False),
    "role_category"
] = "Business Analyst"

df.loc[
    df["job_role"].str.contains("analytics engineer", case=False, na=False),
    "role_category"
] = "Analytics Engineer"

In [10]:
df["role_category"].value_counts().head(15)

role_category
Data Engineer               3315
Data Scientist              1679
Data Analyst                1030
Business Analyst             674
BI Engineer                   83
Big Data Developer            65
Senior Software Engineer      51
Product Analyst               43
Python Developer              42
Software Engineer             36
Analyst                       35
Senior Analyst                33
Data Architect                30
Research Analyst              30
Analytics Engineer            29
Name: count, dtype: int64

In [11]:
# Salary bins
salary_bins = [0, 500000, 1000000, 1500000, 2000000, float("inf")]

salary_labels = [
    "<5 LPA",
    "5-10 LPA",
    "10-15 LPA",
    "15-20 LPA",
    ">20 LPA"
]

# Create salary category
df["salary_category"] = pd.cut(
    df["salary"],
    bins=salary_bins,
    labels=salary_labels
)

In [12]:
df["salary_category"].value_counts(dropna=False)

salary_category
>20 LPA      2575
15-20 LPA    2509
10-15 LPA    2473
5-10 LPA     2459
<5 LPA       1008
NaN           976
Name: count, dtype: int64

In [13]:
# Number of skills mentioned in each job
df["skill_count"] = (
    df["skills_description"]
    .str.split(",")
    .str.len()
)

In [15]:
df[["skills_description", "skill_count"]].head(20)

,skills_description,skill_count
0,"python, MLT, statistical modeling, machine lea...",8
1,"Data Science, Machine learning, Python, Azure,...",8
2,"Python, IT Skills, Testing, Cloud, Product Man...",8
3,"python, machine learning, Data Science, data a...",6
4,"IT Skills, Python, Data Science, Machine Learn...",8
5,"Data Science, oral communication, Shell, Pytho...",8
6,"Computer science, cassandra, Machine learning,...",8
7,"team lead, MLT, machine learning, aws, Python,...",8
8,"Graphics, Bidding, Google Analytics, Data mana...",8
9,"Machine Learning, Python, Tableau, Azure, GCP,...",8


In [18]:
# Create experience level
df["experience_level"] = "Senior"

df.loc[df["exp_avg"] <= 1, "experience_level"] = "Fresher"
df.loc[(df["exp_avg"] > 1) & (df["exp_avg"] <= 3), "experience_level"] = "Junior"
df.loc[(df["exp_avg"] > 3) & (df["exp_avg"] <= 5), "experience_level"] = "Mid-Level"

In [20]:
df["experience_level"].value_counts()

experience_level
Senior       4667
Mid-Level    3911
Junior       2868
Fresher       554
Name: count, dtype: int64

In [21]:
df.to_csv(
    "../data/processed/naukri_data_science_jobs_india_processed.csv",
    index=False
)